In [20]:
# 사람 개입 기능.
# 다음 상황에서는 사람의 개입이 필요할 수있음
#   . 중용한 결정을 내려야 할때.
#   . AI의 판단이 불확실할때 위험할 때.
#   . 사람의 검토나 승인 없이 실행해서는 안되는 작업.

# 이러한 경우 langgraph에서는 사람이 직접 개입할 수 있는 사람개입(Human In The Loop: HITL)기능을 제공함.
# 이 기능을 사용하면 챗봇의 실행 흐름을 중단하고 사용자의 피드백이나 판단을 기다렸다가 다시 흐름을 이어갈 수있음.

# 1) interrupt() 함수란?
#   interrupt 함수는 마치 파이썬의 input() 처럼 작동함.
#   이 함수를 호출하면 그래프의 실행이 일시 중단돠며, 외부에서 전달된 Command 객체를 통해 다시 흐름을 이어갈 수 있음.

#   01. 사람의 도움을 요청하는 도구 정의하기.
#   먼저, 사람이 판단이 필요한 상황에서 호출할 도구인 human_assistance 를 정의함.
#   이 도구는 내부적으로 interrupt() 함수를 호출해 그래프 실행을 일시 중지시키고, 외부로부터 사용자의 입력을 기다리는 상태로 전환됨.

from langgraph.types import Command, interrupt
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model
from langchain_tavily import TavilySearch

llm = init_chat_model("openai:gpt-4.1")

@tool # 데코레이터
def  human_assistance(query: str):
    """사람의 도움이 필요할때 호출하는 도구입니다."""
    human_response = interrupt({"query": query}) # 실행 시 일시 중단
    return human_response

# 위 코드에서는 먼저 LLM을 초기화하고, @tool 데코레이터를 사용해 일반 파이썬 함수를 랭그래프에서 사용할 수있는 도구로 등록함.
# 이렇게 등록된 도구는 챗봇이 필요하다고 판단할 경우 자동으로 호출할 수 있음.
# 이때 랭그래프는 현재 실행중인 흐름을 일시적으로 중단하고, 도구 실행에 필요한 사용자 응답을 받은 뒤, 그 결과를 바탕으로 다시 그래플 실행을 이어감.

In [21]:
#   02. 도구 목록에 추가.
#   앞에서 사용한 웹 검색 도구인 TavilySearch와 함께 human_assistance 도구를 추가함.
#   이렇게 하면 챗봇이 판단이 어려운 상황에서 사람이 개입할 수 있도록 도와줌.
#   그리고 bind_tools()로 LLM과 도구를 연결함.
search_tool = TavilySearch(max_result=2)
tools = [search_tool, human_assistance]
llm_with_tools = llm.bind_tools(tools)

In [22]:
#   03. 상태 및 챗봇 노드 정의
#   랭그래프 관련 모듈을 임포트하고, 챗봇의 핵심 동작을 담당할 상태와 챗봇 노드를 정의함.
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages

# 상태 정의
class State(TypedDict):
    messages: Annotated[list, add_messages]

# 챗봇 노드 정의
def chatbot(state:State):
    message = llm_with_tools.invoke(state["messages"])
    return {"messages": [message]}

In [23]:
#   04. 그래프 구성.
#   이제 사람 개입이 가능한 도구를 포함한 전체 그래프 구성.

# 그래프 초기화 및 메모리 설정.
graph_builder = StateGraph(State)
memory = MemorySaver()

# 노드 및 엣지 추가
graph_builder.add_node("chatbot", chatbot)
tool_node = ToolNode(tools=tools)
graph_builder.add_node("tools", tool_node)
graph_builder.add_conditional_edges("chatbot", tools_condition) # 에이전트가 워크플로우를 수행할때, "다음에 도구를 호출해야하는지, 아니면 사용자에게 최종 답변을 해야 하는지" 자동으로 판단해 주는 
graph_builder.add_edge("tools", "chatbot")                     # 내장 조건부 라우팅(Conditional Routing) 함수임
graph_builder.add_edge(START, "chatbot")

# 그래프 컴파일.
graph  = graph_builder.compile(checkpointer = memory)

# 이 챗봇은 필요할 때 사람의 판단을 요청할 수 있으며, 사용자가 입력을 제공하면 그 이후 흐름을 계속 이어감.

#   05. 질문 입력으로 챗봇 시작하기.
#   이제 human_assistance도구가 실제러 어떻게 동작하는지, 그리고 어떻게 사람의 판단으 반영해 중단된 실행을 이어갈수 있는지 살펴 보겠음.
#   우선, human_assistance 도구가 호출 될 수 있도록 다음과 같이 입력을 전달함.
user_input = "AI 에이전트를 만들고 싶은데 전문가의 조언이 필요해요. 사람의 도움을 받을 수 있을까요?"
config = {"configurable": {"thread_id": 1}}

events = graph.stream(
    {"messages": [{"role":"user", "content": user_input}]},
    config,
    stream_mode = "values"
)

for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

================================ Human Message =================================

AI 에이전트를 만들고 싶은데 전문가의 조언이 필요해요. 사람의 도움을 받을 수 있을까요?
================================== Ai Message ==================================
Tool Calls:
  human_assistance (call_ouWGtAoRDDIcxkr9wufDETIr)
 Call ID: call_ouWGtAoRDDIcxkr9wufDETIr
  Args:
    query: AI 에이전트를 만들고 싶은데 전문가의 조언이 필요합니다. 사람의 도움을 받을 수 있을까요?
================================== Ai Message ==================================
Tool Calls:
  human_assistance (call_ouWGtAoRDDIcxkr9wufDETIr)
 Call ID: call_ouWGtAoRDDIcxkr9wufDETIr
  Args:
    query: AI 에이전트를 만들고 싶은데 전문가의 조언이 필요합니다. 사람의 도움을 받을 수 있을까요?


In [24]:
# 실행결과를 보면 챗봇이 human_assistance 도구를 호출하고 중단된 것을 확인할 수 잇음.
# 그래프의 상태를 확인해 보면 현재 실행 위치가 ("tools",)에 멈춰 있음을 알수 있음. 
snapshot = graph.get_state(config)
print(snapshot.next)

('tools',)


In [ ]:
#   06. 실행 재개하기
#   사람이 판단이나 입력이 준비됐다면 Command(resume=...)를 사용해 그래프 실행을 재개할수 있음.
#   이때 resume에 전달한 값은 노드(또는 도구)내부에서 호출된 interrupt() 의 반환값으로 그대로 전달됨.
#   따라서 반드시 {"data":...}처럼 특정 딕셔너리 형태를 고정할 필요는 없으며, 문자열이나 객체 등 JSON 직렬화 가능한 값이라면 원하는 형태로 전달할 수 있음.
from langgraph.types import Command

human_response = (
    "전문가입니다. LangGraph를 활용해 AI 에이전트를 만들어 보세요.",
    "간단한 자율 에이전트보다 훨씬 더 확장성 있고 안정적입니다."
)

human_command = Command(resume={"data": human_response})

events = graph.stream(human_command, config, stream_mode="values")
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

# 이제 그래프는 이전 중단 지점부터 이어서 실행되며, human_assistance 도구의 결과를 받아 응답을 마무리함.
# 이전에 중단된 상태가 체크포인터에 의해 저장되어 있었기 때문에 중단 이후에도 자연스럽게 대화를 이어갈 수 있음.

# 중단된 흐름을 다시 이어주는 checkpointer
# 상태가 저장돼 있기 때문에 중간에 중단되더라도 언제든지 이어서 실행할 수 잇음.
# 이는 단순한 메모리 저장을 넘어서 사람의 판단을 기다리는 워크플로우나 승인 기반 흐름에도 유용하게 활용할 수 있음.